In [3]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, save_img
import warnings
warnings.filterwarnings('ignore')

# Paths to your original dataset directories (input)
input_class_paths = {
    'VeryMildDemented': r"C:\Users\Sai charan\Desktop\pr_mini_project\Alzheimer_s Dataset\VeryMildDemented",
    'MildDemented': r"C:\Users\Sai charan\Desktop\pr_mini_project\Alzheimer_s Dataset\MildDemented",
    'NonDemented': r"C:\Users\Sai charan\Desktop\pr_mini_project\Alzheimer_s Dataset\NonDemented",
    'ModerateDemented': r"C:\Users\Sai charan\Desktop\pr_mini_project\Alzheimer_s Dataset\ModerateDemented"
}

# Paths to where the augmented images will be saved (output)
output_class_paths = {
    'VeryMildDemented': r"C:\Users\Sai charan\Desktop\balanced_augmentation\VeryMildDemented",
    'MildDemented': r"C:\Users\Sai charan\Desktop\balanced_augmentation\MildDemented",
    'NonDemented': r"C:\Users\Sai charan\Desktop\balanced_augmentation\NonDemented",
    'ModerateDemented': r"C:\Users\Sai charan\Desktop\balanced_augmentation\ModerateDemented"
}

# Number of images we want for each class
target_images = 3200

# Create an instance of the ImageDataGenerator with the desired augmentations
datagen = ImageDataGenerator(
    rotation_range=15,       # Small rotations between 0° to 15°
    width_shift_range=0.1,   # Small horizontal shifts (10% of width)
    height_shift_range=0.1,  # Small vertical shifts (10% of height)
    shear_range=0.1,         # Small shear transformations
    zoom_range=0.1,          # Small zoom in/out (10%)
    horizontal_flip=True,    # Horizontal flip
    vertical_flip=False,     # Vertical flip (set to False to prevent excessive changes)
    brightness_range=[0.8, 1.2],  # Brightness adjustments (reduce by 20%, increase by 20%)
    fill_mode='nearest'      # Fill mode for handling borders of images
)

# Assuming you have loaded your dataset as numpy arrays X_train (images) and y_train (labels)

# Apply the augmentations to your dataset in batches


# Use the augmented data for training
# Example: If using Keras for a model
# model.fit(augmented_data, epochs=10, steps_per_epoch=len(X_train) // batch_size)


# Function to filter out non-image files and augment images
def is_image_file(file_name):
    return file_name.lower().endswith(('.png', '.jpg', '.jpeg'))

# Function to augment images and save them to the output folder
def augment_class_images(class_name, input_class_path, output_class_path):
    # List only image files from the directory
    images = [f for f in os.listdir(input_class_path) if is_image_file(f)]
    num_existing_images = len(images)
    
    if num_existing_images >= target_images:
        print(f"{class_name} already has {num_existing_images} images. Skipping augmentation.")
        return
    
    print(f"Augmenting {class_name}: {num_existing_images} -> {target_images}")
    
    # Create output directory if it doesn't exist
    os.makedirs(output_class_path, exist_ok=True)
    
    # Calculate how many more images we need
    num_to_generate = target_images - num_existing_images
    
    for i in range(num_to_generate):
        # Randomly select an image from the input folder
        img_name = np.random.choice(images)
        img_path = os.path.join(input_class_path, img_name)
        
        # Load and prepare the image
        img = load_img(img_path)
        x = img_to_array(img)
        x = np.expand_dims(x, axis=0)
        
        # Generate batches of augmented images
        it = datagen.flow(x, batch_size=1)
        batch = next(it)
        augmented_image = batch[0].astype('uint8')
        
        # Save the augmented image to the output folder with a unique name
        new_img_name = f"aug_{i}_{os.path.splitext(img_name)[0]}.png"  # Save as PNG for consistency
        save_img(os.path.join(output_class_path, new_img_name), augmented_image)

# Augment images for each class and save them in separate output directories
for class_name in input_class_paths.keys():
    augment_class_images(
        class_name,
        input_class_paths[class_name],
        output_class_paths[class_name]
    )


Augmenting VeryMildDemented: 2240 -> 3200
Augmenting MildDemented: 896 -> 3200
NonDemented already has 3200 images. Skipping augmentation.
Augmenting ModerateDemented: 64 -> 3200
